# Visualizing Networkx output via vmd

In [ ]:
from community import community_louvain
import community as c
from collections import Counter
from collections import defaultdict
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCWriter
import mdtraj as md
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import pickle
import networkx as nx
import random
import matplotlib.pyplot as plt
import random
import pickle

In [ ]:
#### Re-transforming these clusters into trajectory snippets
def convert_to_tcl(residue_dict, cluster_id):
    if cluster_id not in residue_dict:
        return ""
    residue_lists = residue_dict[cluster_id]
    tcl_res_list = "set res_list {\n"
    for res_tuple in residue_lists:
        tcl_res_list += f"    {{{' '.join(map(str, res_tuple))}}}\n"
    tcl_res_list += "}\n"

    return tcl_res_list

In [ ]:
alpha_full_length = "alpha_syn_full_peptide"
medin_cm14 = 'medin_cm14'
medin_cm8 = 'medin_cm8'
abeta = "abeta"
medin_urea = 'medin_urea'
uplet_type=5

protein_name = medin_cm8

In [ ]:
if protein_name == 'abeta':
    pdb = '/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/template_G5.pdb'
    xtc = '/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/traj_all-skip-0-noW_G5.xtc'
    w_file = "/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/abeta/weights/weights_corr_G5"
    distances_com = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/abeta/distances/d_24_t_com_avg.pkl', 'rb'))
    distances_closest = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/abeta/distances/d_24_t_com_avg.pkl', 'rb'))
    w_eq = np.loadtxt(w_file)
    resname_ligand = "liga"
    special_char = 'A\u03B2'
    w_com = 0.8
    w_closest = 0.2

elif protein_name == 'medin_cm14':
    pdb = '/Users/adelielouet/Documents/science/medin/cm14/simu/prepared_system/CM14/prepared_tpr_files_v1/template_complex.gro'
    xtc = '/Users/adelielouet/Documents/science/medin/cm14/simu/prepared_system/CM14/prepared_tpr_files_v1/cat_trjcat.xtc'
    resname_ligand = "1UNL"
    special_char = 'Medin_cm14'
    w_com = 0.45
    w_closest = 0.55
    w_eq=np.array([1.0] *len(distances_com))   # this is for systems that don't need to be reweighted, assign each frame a weight of 1.0

elif protein_name == 'medin_cm8':
    pdb = '/Users/adelielouet/Documents/science/medin/cm8/cm8_post_run/protein_ligand.gro'
    xtc = '/Users/adelielouet/Documents/science/medin/cm8/cm8_post_run/concatenate_cm8_skip_10.xtc'
    w_file='/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/weights/COLVAR_REWEIGHT'
    distances_com = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/distances/d_24_t_com_avg.pkl', 'rb'))
    distances_closest = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/distances/d_24_t_closest.pkl', 'rb'))
    resname_ligand = "1UNL"
    special_char = 'Medin_cm8'
    w_com = 0.45
    w_closest = 0.55
    w_eq=np.array([1.0] *len(distances_com))

elif protein_name == 'alpha_syn_full_peptide':
    pdb = '/Users/adelielouet/Documents/science/alpha_syn/DE_SHAW/system.pdb'
    # xtc = ???  # xtc path appears to be missing for alpha
    resname_ligand = "*"
    special_char = "\u03B1-syn-full"
    w_com = 0.1
    w_closest = 0.9
    w_eq=[1.0] *len(distances_com)   # this is for systems that don't need to be reweighted, assign each frame a weight of 1.0

elif protein_name == 'medin_urea':
    pdb = '/Users/adelielouet/Documents/science/medin/urea/urea_post_run/protein_urea.gro'
    xtc = '/Users/adelielouet/Documents/science/medin/urea/urea_post_run/concatenate_urea.xtc'
    resname_ligand = "1UNL"
    distances_com = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_urea/distances/d_24_t_com_avg.pkl', 'rb'))
    distances_closest = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_urea/distances/d_24_t_closest.pkl', 'rb'))
    w_file='/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/weights/COLVAR_REWEIGHT'
    special_char = 'Medin_urea'
    w_com = 0.45
    w_closest = 0.55
    w_eq=np.array([1.0] *len(distances_com))

else:
    pass


The following stored dictionaries come from running clustering_uplets.ipynb & p_eq_weighted.ipynb

In [ ]:
%store -r inv_map
%store -r unique_uplets_pre_process_0


The following code navigates through a directory and generates a separate VMD (Tcl) script for each cluster. It requires two directories:

clustered_traj_xtc: to store the clustered trajectory files.

clustered_tcl_scripts: to store the corresponding VMD Tcl scripts.

The first part of the code extracts trajectory frames corresponding to each cluster from the full-length trajectory and concatenates them into the clustered_traj_xtc directory.

The second part generates a Tcl script for each cluster to visualize its trajectory in VMD. You can run these scripts using the following terminal command:

bash
Copy
Edit
vmd -e cluster_X.tcl
Replace X with the appropriate cluster number.



In [ ]:
u = mda.Universe(pdb, xtc)

inv_map[-1]=[tuple([-1]*uplet_type)]

tuple_to_key = {}
for key, value_list in inv_map.items():
    for tup in value_list:
        tuple_to_key[tuple(sorted(tup))] = key  # Sort and store as tuple

inv_map_vals=[]
for keys,values in inv_map.items():
    inv_map_vals.append(values)

matched_keys = [tuple_to_key.get(tuple(sorted(lst)), None) for lst in unique_uplets_pre_process_0]


cluster_writers = {}

for ts, cluster_id in zip(u.trajectory, matched_keys):
    if cluster_id != -1:
        if cluster_id not in cluster_writers:
            output_filename = f"/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/visualization/{protein_type}/clustered_traj_xtc/cluster_{cluster_id}.xtc"
            cluster_writers[cluster_id] = XTCWriter(output_filename, n_atoms=u.atoms.n_atoms)

        cluster_writers[cluster_id].write(u)

for writer in cluster_writers.values():
    writer.close()

for cluster_id,res in zip(inv_map.keys(),inv_map.values()):
    tcl_filename = f"/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/visualization/{protein_type}/clustered_tcl_scripts/cluster_{cluster_id}.tcl"

    tcl_residues = convert_to_tcl(inv_map, cluster_id)

    # Write the TCL script
    with open(tcl_filename, 'w') as tcl_file:
        tcl_file.write(f"""
                       
# TCL script for Cluster {cluster_id}

# Load PDB structure and XTC trajectory
mol new "{pdb}"
mol addfile "/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/visualization/{protein_type}/clustered_traj_xtc/cluster_{cluster_id}.xtc" type xtc waitfor all

# Remove all solvent and ions (non-protein and non-LIGA residues)
set solvent_ions [atomselect top "not protein and not resname {resname_ligand}"]
$solvent_ions delete

# Remove all representations (clear any previous representations)
mol delete rep all

# Show protein as ribbons (NewCartoon representation), remove lines underneath
mol representation Ribbons
mol color Name
mol selection "protein"
mol material Glossy
mol width 4
mol addrep top

# Ensure no bonds or lines under the protein (turn off bond representation)
mol representation None
mol addrep top

# Color and display selected residues in red (VDW representation, RGB color)
{tcl_residues}

# Apply red color and VDW representation for each selected residue (only spheres)
foreach res_group $res_list {{
    set res_str ""
    foreach res_id $res_group {{
        set res_str "${{res_str}} resid $res_id or"
    }}
    # Remove the last "or"
    set res_str [string trimright $res_str "or"]

    # Select the residues and apply VDW representation with RGB red color
    set sel [atomselect top "$res_str"]
    mol representation VDW 0.5 12.0  # Set VDW sphere size and transparency (alpha 0.5)
    mol color ColorID 1  # RGB for red color in VDW
    mol selection "$res_str"
    mol addrep top
}}

set liga [atomselect top "resname {resname_ligand}"]
mol representation Bonds
mol color ColorID 0  # Ligand bonds will be blue
mol selection "resname {resname_ligand}"
mol addrep top

display background white
color Display Background white
axes off

# Update the display and reset view
display resetview

""")


### RMSD Calculation (Intra/Inter - cluster)

Histograms showing the probability density distributions of the radius of gyration (Rg) and root-mean-square deviation (RMSD) across 5 arbitrary clusters.

In [ ]:

rmsd_data = {}
rg_data = {}
rmsd_stats = {}
rg_stats = {}

cluster_xtc_dir = f"/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/visualization/{protein_name}/quintuplets/clustered_traj_xtc"
ref=md.load(pdb)
#ref = md.load_pdb(pdb)
num_clusters = len(inv_map_vals)
blues = sns.color_palette("crest", n_colors=6)
plt.figure(figsize=(12, 6))

random_selec=random.sample(range(1, num_clusters), 5)

cluster_color=0
# Plotting RMSD and Rg distributions for each cluster
for cluster in random_selec:
    cluster_color+=1
    cluster_xtc_path = os.path.join(cluster_xtc_dir, f"cluster_{cluster}.xtc")
    traj = md.load(cluster_xtc_path, top=pdb)
    topology_pro = traj.topology.select('protein')

    rmsd = md.rmsd(traj, ref, frame=0, atom_indices=topology_pro)
    rg = md.compute_rg(traj)

    rmsd_data[cluster] = rmsd
    rg_data[cluster] = rg

    rmsd_stats[cluster] = (np.mean(rmsd), np.std(rmsd))
    rg_stats[cluster] = (np.mean(rg), np.std(rg))

    # Plot RMSD
    plt.subplot(1, 2, 1)
    sns.kdeplot(rmsd, shade=True, alpha=0.6, color=blues[cluster_color], label=f"Cluster {cluster_color}")

    # Plot Rg
    plt.subplot(1, 2, 2)
    sns.kdeplot(rg, shade=True, alpha=0.6, color=blues[cluster_color], label=f"Cluster {cluster_color}")


# Intra-cluster stats
avg_rmsd_intra = np.mean([std for _, std in rmsd_stats.values()])
avg_rg_intra = np.mean([std for _, std in rg_stats.values()])

# Inter-cluster differences
cluster_pairs = [(i, j) for i in range(num_clusters) for j in range(i + 1, num_clusters)]
valid_pairs = [(i, j) for i, j in cluster_pairs if i in rmsd_stats and j in rmsd_stats]

rmsd_inter_diffs = [abs(rmsd_stats[i][0] - rmsd_stats[j][0]) for i, j in valid_pairs]
rg_inter_diffs = [abs(rg_stats[i][0] - rg_stats[j][0]) for i, j in valid_pairs]

avg_rmsd_inter = np.mean(rmsd_inter_diffs)
avg_rg_inter = np.mean(rg_inter_diffs)

# Final plot formatting
plt.subplot(1, 2, 1)
plt.xlabel("RMSD (nm)", fontsize=14)
plt.ylabel("Density", fontsize=14)

plt.subplot(1, 2, 2)
plt.xlabel(r'$\mathit{Rg\ (nm)}$', fontsize=14)
#plt.xlabel("Radius of Gyration (nm)", fontsize=14)
plt.ylabel("Density", fontsize=14)

#plt.suptitle(f"{special_char} Cluster Evaluation", fontsize=20)

# Add summary statistics below the plot
plt.figtext(0.10, -0.05, f"RMSD inter-cluster mean diff: {avg_rmsd_inter:.3f} nm", fontsize=13)
plt.figtext(0.10, -0.09, f"RMSD intra-cluster std avg: {avg_rmsd_intra:.3f} nm", fontsize=13)
plt.figtext(0.55, -0.05, f"Rg inter-cluster mean diff: {avg_rg_inter:.3f} nm", fontsize=13)
plt.figtext(0.55, -0.09, f"Rg intra-cluster std avg: {avg_rg_intra:.3f} nm", fontsize=13)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
#save_path = f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/figures/{protein_name}_cluster_eval.png'
plt.savefig(save_path, dpi=300)
plt.show()